In [1]:
import pvdeg

import xarray as xr
import pandas as pd
from dask.distributed import LocalCluster, Client
import glob
import numpy as np
import dask.array as da
import matplotlib.pyplot as plt

#import inspire_agrivolt
from pathlib import Path

In [13]:
zarrs = {}

for conf in ["01","02","03","04","05","06","07","08","09","10", "11"]:
    print(conf)
    
    final_dir = Path("/projects/inspire/PySAM-MAPS/v1.2/final-backup/")
    conf_zarr_path = final_dir / f"{conf}.zarr"
    
    conf_zarr = xr.open_zarr(str(conf_zarr_path))
    zarrs[conf] = conf_zarr
    
    # for var in conf_zarr.data_vars:
    #     print(conf, var, conf_zarr[var].isnull().values.any())

01
02
03
04
05
06
07
08
09
10
11


In [17]:
for name, zarr in zarrs.items():
    if "distances_m" in zarr.data_vars:
        print("found distances_m in zarr", name)
        
        dm = zarr.distances_m

        # false means there are only finite values in the dataset
        nonfinite = (~xr.apply_ufunc(np.isfinite, dm, dask='allowed')).any().compute().item()
        dm_min = dm.min().compute().item()
        dm_max = dm.max().compute().item()
        
        print(name, "nonfinite?", nonfinite, "min", dm_min, "max", dm_max)

found distances_m in zarr 01
01 nonfinite? False min 0.25 max 4.75
found distances_m in zarr 02
02 nonfinite? False min 0.25 max 4.75
found distances_m in zarr 03
03 nonfinite? False min 0.25 max 4.75
found distances_m in zarr 04
04 nonfinite? False min 0.4 max 7.6
found distances_m in zarr 05
05 nonfinite? False min 0.5555555555555556 max 10.555555555555555
found distances_m in zarr 06
06 nonfinite? False min 0.19 max 7.097951532344773
found distances_m in zarr 07
07 nonfinite? False min 0.19 max 7.097951532344773
found distances_m in zarr 08
08 nonfinite? False min 0.19 max 7.097951532344773
found distances_m in zarr 09
09 nonfinite? False min 0.38 max 14.195903064689546
found distances_m in zarr 10
10 nonfinite? False min 0.43478260869565216 max 8.26086956521739
found distances_m in zarr 11
11 nonfinite? False min 0.19 max 7.097951532344773


In [ ]:
def ground_irradiance_distances(ds: xr.Dataset) -> xr.DataArray:
    """
    Calculate beds distances from pitch.

    TAKEN FROM beds_postprocessing
    """
    NUM_BEDS = 10
    pitch = ds.pitch

    frac = xr.DataArray(
        (np.arange(NUM_BEDS) + 0.5) / NUM_BEDS,
        coords={"distance": np.arange(10, dtype=np.int32)},
        name=f"{NUM_BEDS}_fraction",
    )

    distances = (pitch * frac).rename("distances_m")
    return distances

In [ ]:
original_01 = zarrs["01"].isel(gid=slice(0,10000)).copy()
original_06 = zarrs["06"].isel(gid=slice(0,10000)).copy()

zarrs["01"].isel(gid=slice(0,10000)).to_zarr("/scratch/tford/test-01.zarr")
zarrs["06"].isel(gid=slice(0,10000)).to_zarr("/scratch/tford/test-06.zarr")

distances_01_m_da = ground_irradiance_distances(zarrs["01"].isel(gid=slice(0,10000)))
distances_06_m_da = ground_irradiance_distances(zarrs["06"].isel(gid=slice(0,10000)))

In [ ]:
distances_01_m_da

In [ ]:
distances_06_m_da

In [ ]:
distances_01_m_da.isel(gid=slice(0,10000)).to_zarr("/scratch/tford/test-01.zarr", mode='a')
distances_06_m_da.isel(gid=slice(0,10000)).to_zarr("/scratch/tford/test-06.zarr", mode='a')

In [ ]:
loaded_01 = xr.open_zarr("/scratch/tford/test-01.zarr")
loaded_06 = xr.open_zarr("/scratch/tford/test-06.zarr")

In [ ]:
loaded_01.distances_m.plot()

In [ ]:
loaded_06.distances_m.plot()

In [ ]:
len(set(loaded_06.pitch.compute().values))

In [ ]:
len(set((tuple(row) for row in loaded_06.distances_m.compute().values.tolist())))

In [ ]:
set(loaded_01.data_vars) - set(loaded_06.data_vars)

In [ ]:
set(loaded_06.data_vars) - set(loaded_01.data_vars)

In [ ]:
assert loaded_01.drop_vars(['distances_m']).equals(original_01)
assert loaded_06.drop_vars(['distances_m']).equals(original_06)

In [ ]:
WEATHER_DB = "NSRDB"
WEATHER_ARG = {
    "satellite": "Americas",
    "names": "TMY",
    "NREL_HPC": True,
    "attributes": pvdeg.pysam.INSPIRE_NSRDB_ATTRIBUTES,
}

geo_weather, geo_meta = pvdeg.weather.get(
    WEATHER_DB, geospatial=True, **WEATHER_ARG
)

In [ ]:
geo_meta[['latitude','longitude']]#.to_csv("nsrdb_full_meta.csv")
#import os

#print(os.getcwd())

In [ ]:
geo_meta[['longitude']].plot()

In [ ]:
conf = "01"
state = "Colorado"
target_path = f"/projects/inspire/PySAM-MAPS/Full-Outputs/{state}/{conf}/merged.zarr"

merged = xr.open_zarr(target_path)

In [ ]:
xr.open_dataset("/projects/inspire/PySAM-MAPS/test-cli/05/462482-545008.nc").albedo.mean("time").plot()

In [ ]:
import 

inspire_agrivolt.verify_dataset_gids

In [ ]:
# confirm that all points are filled in
# might be worth checking all variables individually 
ref_var = "albedo" # comes from nsrdb -> pysam -> outputs

# this gets us pairs where null
mask = merged_with_gids[ref_var].mean(dim="time").isnull()

skipped_gids = merged_with_gids.where(mask.compute(), drop=False).gid

# visual representation of what was skipped
#skipped_gids.plot()

gids_arr = skipped_gids.values
gids_arr = gids_arr[~np.isnan(gids_arr)].astype(int)

gids_arr

In [ ]:
@pvdeg.decorators.geospatial_quick_shape("numeric", ("gid",))
def map_gid(weather_df, meta):
    return meta["gid"]
    

In [ ]:
np.genfromtxt(f"states-gids/{state}-gids.txt", dtype=float)

In [ ]:
merged

In [ ]:
workers = 8

cluster = LocalCluster(
    n_workers=workers,
    processes=True,
    dashboard_address=22118,
)

client = Client(cluster)

print(client.dashboard_link)

In [ ]:
conf = "10"
files = glob.glob(f"/projects/inspire/PySAM-MAPS/Full-Outputs/Colorado/{conf}/*.nc")
zarr_path = f"/projects/inspire/PySAM-MAPS/Full-Outputs/Colorado/{conf}/merged.zarr"

inspire_agrivolt.pysam_output_netcdf_to_zarr(
    files=files,
    zarr_path=zarr_path
)

In [ ]:
full_co_01 = xr.open_zarr(zarr_path)

In [ ]:
full_co_01.pitch.plot()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import get_cmap

def plot_dataset_grids(files: list[str], engine="netcdf4") -> None:
    """
    Plot the (longitude, latitude) grid points from each dataset with different colors.

    Parameters
    ----------
    files : list[str]
        List of NetCDF file paths.
    engine : str
        Backend engine to use for opening NetCDF files.
    """
    cmap = get_cmap("tab20")  # Up to 20 distinguishable colors
    num_colors = len(files)

    plt.figure(figsize=(10, 6))

    for i, file in enumerate(files):
        with xr.open_dataset(file, engine=engine) as ds:
            lats = ds.latitude.values
            lons = ds.longitude.values

            grid_lon, grid_lat = np.meshgrid(lons, lats)

            plt.scatter(
                grid_lon.flatten(),
                grid_lat.flatten(),
                s=3,
                color=cmap(i % 20),
                label=f"File {i}",
                marker='x'
            )

    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Grid points from each NetCDF file")
    plt.legend(markerscale=3, loc='upper right', bbox_to_anchor=(1.15, 1))
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_dataset_grids(files)

In [ ]:
xr.open_zarr(zarr_path).pitch.plot()

In [ ]:
WEATHER_DB = "NSRDB"
WEATHER_ARG = {
    "satellite": "Americas",
    "names": "TMY",
    "NREL_HPC": True,
    "attributes": pvdeg.pysam.INSPIRE_NSRDB_ATTRIBUTES,
}

In [ ]:
weather, meta, chunk_size = inspire_agrivolt.load_weather(local_test_paths=None, state="Colorado")

In [ ]:
weather

In [ ]:
pvdeg.geospatial.output_template

In [ ]:
pvdeg.geospatial.output_template(ds_gids=weather, shapes=pvdeg.pysam.INSPIRE_GEOSPATIAL_TEMPLATE_SHAPES, add_dims={"distance":10})

In [ ]:
meta.iloc[12:15].index

In [ ]:
pvdeg.weather.get(
    WEATHER_DB,
    geospatial=True, 
    **WEATHER_ARG
)

In [ ]:
len(meta)

In [ ]:
"wind_direction" in geo_weather.data_vars

In [ ]:
geo_weather = geo_weather.assign(wind_direction=geo_weather["temp_air"] * 0)
geo_weather = geo_weather.assign(albedo=geo_weather["temp_air"] * 0 + 0.2) 

geo_weather

In [ ]:
geo_weather = xr.open_dataset("C:/Users/tford/Downloads/small-usa-tmy.nc")
geo_meta = pd.read_csv("C:/Users/tford/Downloads/small-usa-tmy.csv", index_col=0)

weather_df = geo_weather.isel(gid=0).to_dataframe()
meta = geo_meta.iloc[0].to_dict()

# add placeholder wind and albedo data
# this will come from the NSRDB but this file does not contain it
weather_df["wind_direction"] = 0
weather_df["albedo"] = 0.2

In [ ]:
conf = "01"

single_loc_res = pvdeg.pysam.inspire_ground_irradiance(
    weather_df=weather_df,
    meta=meta,
    config_files={"pv":f"C:/Users/tford/dev/InSPIRE/Studies/USMap_Doubleday_2024/SAM/{conf}/{conf}_pvsamv1.json"}
)

In [ ]:
# 15 day plot we can see that they all have values
single_loc_res.ground_irradiance.isel(time=slice(650,1000)).plot()